# fabric-medallion-dbt

One step of `run_pipeline`, which is what you run or schedule:

```
land  ->  dwh   ->  parity
      ->  spark ->
```

`land` downloads the AEMO files once, `dwh` and `spark` build the same gold layer on Fabric
Warehouse and Fabric Spark, and `parity` fails the run if the two disagree.

Every step is the same four moves: read the settings, fetch the project, install what the
step needs, run the step. dbt is not preinstalled in a Fabric notebook; the install cell
pip-installs the adapter from the project's `requirements/`.

Settings are in the `deploy_config` Variable Library: `engines` (`all`, `dwh` or `spark`),
the download and process limits, and where the project comes from. `project_source` is
`github` (the public [repo](https://github.com/djouallah/fabric-medallion-dbt) at
`repo_ref`, a branch or tag) or `onelake` (the copy a deploy from CI left in `dbt_landing`).

In [ ]:
# Set by run_pipeline. Run by hand, this lands the files; change `step` to build an engine.
step = "land"       # land | dwh | spark | parity
run_id = "manual"

In [ ]:
# 1. SETTINGS: the deploy_config Variable Library, as the env vars the scripts and the dbt
# profile read.
import io
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import notebookutils

ENGINES = ("dwh", "spark")
STEPS = ("land", *ENGINES, "parity")
if step not in STEPS:
    raise ValueError(f"step={step!r}: expected {' | '.join(STEPS)}")

vl = notebookutils.variableLibrary.getLibrary("deploy_config")
for k in ("engines", "process_limit", "download_limit", "daily_download_limit"):
    os.environ[k] = getattr(vl, k)
os.environ["DBT_SCHEMA"] = vl.dbt_schema
os.environ["SPARK_RESOURCE_PROFILE"] = vl.spark_resource_profile
os.environ["FABRIC_WORKSPACE_ID"] = notebookutils.runtime.context["currentWorkspaceId"]
# The workspace folder the install created; provision.py keeps the lakehouses and the
# warehouse in it, next to this notebook.
os.environ["FOLDER"] = "fabric-medallion-dbt"
# dbt-fabricspark 1.13.x runs OPTIMIZE after every Delta build, which rewrites the layout
# the run just produced.
os.environ["DBT_FABRICSPARK_SKIP_OPTIMIZE"] = "true"
# So the scripts' output reaches the cells below line by line, not in blocks.
os.environ["PYTHONUNBUFFERED"] = "1"

# The engines this run builds. A leg for an engine nobody asked for has nothing to do, and
# neither has parity with one engine.
selected = list(ENGINES) if vl.engines == "all" else [vl.engines]
if not set(selected) <= set(ENGINES):
    raise ValueError(f"engines={vl.engines!r}: expected all | {' | '.join(ENGINES)}")
if (step in ENGINES and step not in selected) or (step == "parity" and len(selected) < 2):
    print(f"{step} skipped: engines={vl.engines}")
    step = None

In [ ]:
# 2. THE PROJECT: the dbt models and the scripts, as one zip, from where `project_source` says.
REPO_URL = "https://github.com/djouallah/fabric-medallion-dbt"
PROJECT_ZIP = "Files/project/fabric-medallion-dbt.zip"

# The ~135 GiB work disk, not the 19 GiB /tmp overlay.
work = "/home/trusted-service-user/work" if os.path.isdir("/home/trusted-service-user/work") else "/tmp"
if vl.project_source == "github":
    # The demo install: the public repo, at `repo_ref`.
    source = f"{REPO_URL}/archive/{vl.repo_ref}.zip"
    with urllib.request.urlopen(source) as r:
        archive = zipfile.ZipFile(io.BytesIO(r.read()))
elif vl.project_source == "onelake":
    # A deploy from CI (.github/scripts/deploy.py) left it in the landing lakehouse, so
    # nothing is fetched from GitHub and the repo can be private.
    landing = notebookutils.lakehouse.get("dbt_landing")
    # notebookutils hands an item back as an object in some runtimes and a dict in others.
    landing_id = landing["id"] if isinstance(landing, dict) else landing.id
    source = (f"abfss://{os.environ['FABRIC_WORKSPACE_ID']}@onelake.dfs.fabric.microsoft.com/"
              f"{landing_id}/{PROJECT_ZIP}")
    local = Path(work, "project.zip")
    local.unlink(missing_ok=True)
    if not notebookutils.fs.cp(source, f"file:{local}"):
        raise RuntimeError(f"could not copy {source}")
    archive = zipfile.ZipFile(local)
else:
    # NO FALLBACK to GitHub: in a private copy that would run the public repo's code
    # against this workspace.
    raise ValueError(f"project_source={vl.project_source!r}: expected github | onelake")

# One top-level folder: <repo>-<ref> from GitHub, fabric-medallion-dbt from a deploy.
repo = os.path.join(work, archive.namelist()[0].split("/")[0])
shutil.rmtree(repo, ignore_errors=True)
archive.extractall(work)
# The zip comment is the commit it was made from, on both routes.
print(f"{source} @ {archive.comment.decode() or vl.repo_ref} -> {repo}")

In [ ]:
# 3. INSTALL what this step needs, from the project's requirements/, the same lists CI
# installs from:
#
#   dwh     requirements/dwh.txt     dbt-fabric
#   spark   requirements/spark.txt   dbt-fabricspark
#   land    requirements/ops.txt     the landing script: DuckDB as a library, the Azure SDKs
#   parity  requirements/ops.txt
#
# ONE ADAPTER PER SESSION. dbt-fabric and dbt-fabricspark shadow each other under
# dbt.adapters and cannot share an environment, which is why run_pipeline gives every step
# its own notebook session. dbt then runs as a subprocess: this kernel never imports it, so
# there is nothing to restart after the install.
def sh(*cmd, check=True):
    """One command in the project folder, its log printed here line by line."""
    p = subprocess.Popen(list(cmd), cwd=repo, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() and check:
        # Raise so a scheduled run goes red instead of silently reporting success.
        raise RuntimeError(f"{shlex.join(cmd)} failed (exit {p.returncode}); see the log above")
    return p.returncode


requirements = {"land": "ops", "dwh": "dwh", "spark": "spark", "parity": "ops"}.get(step)
if requirements:
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"requirements/{requirements}.txt")

In [ ]:
# 4. RUN THE STEP, in the three cells below. The commands are the ones pipeline.yml runs on
# a GitHub runner; what differs is only what a runner did around them.
def provision(what):
    """Create the Fabric items if they are missing. provision.py prints KEY=value lines, the
    paths and ids the dbt profile reads; on a runner they went to $GITHUB_ENV, here they go
    into this session."""
    p = subprocess.run([sys.executable, ".github/scripts/provision.py", what], cwd=repo,
                       capture_output=True, text=True)
    print(p.stderr, end="")
    if p.returncode:
        raise RuntimeError(f"provision.py {what} failed (exit {p.returncode})")
    os.environ.update(dict(kv.split("=", 1) for kv in p.stdout.splitlines() if "=" in kv))


def dbt(*args):
    # `python -m`, not the `dbt` executable: pip may put that outside the notebook's PATH.
    return [sys.executable, "-m", "dbt.cli.main", *args, "--target", step, "--profiles-dir", "."]


def fingerprint(engine):
    """Files/parity/<run_id>/<engine>.json in the landing lakehouse. On a runner the
    fingerprints were handed over as artifacts; here both legs leave theirs in OneLake."""
    return f"{os.environ['LANDING_PATH']}/parity/{run_id}/{engine}.json"

In [ ]:
# LAND, in this step and nowhere else. The landing script rewrites
# csv_raw_archive_log.parquet in place, so two legs landing at once would race on that one
# file; and one landing zone for both engines is what makes parity mean anything.
if step == "land":
    provision("landing")
    sh(sys.executable, "ingest/download_aemo.py")

In [ ]:
# BUILD one engine, test it, fingerprint its gold table.
if step in ENGINES:
    provision(step)
    # provision.py emits CLI, for the runner it was written for. In a notebook each adapter
    # asks notebookutils for the token, and each has its own name for that.
    os.environ["FABRIC_AUTH"] = {"dwh": "notebookutils", "spark": "fabric_notebook"}[step]

    # No --select: what is built is decided by +enabled in dbt_project.yml, on the target.
    if sh(*dbt("build"), check=False):
        sh(*dbt("retry"))

    # Only after a green build: the run-operation reads the gold TABLE, not the run, so taken
    # after a failure it would report the previous run's numbers.
    with tempfile.TemporaryDirectory() as tmp:
        capture = [sys.executable, ".github/scripts/parity.py", "capture", tmp]
        sh("bash", "-o", "pipefail", "-c",
           f"{shlex.join(dbt('run-operation', 'parity_fingerprint'))} | {shlex.join(capture)}")
        notebookutils.fs.put(fingerprint(step), Path(tmp, f"{step}.json").read_text(), True)

In [ ]:
# PARITY, the acceptance test: the two gold tables must agree.
if step == "parity":
    provision("landing")
    with tempfile.TemporaryDirectory() as tmp:
        # THIS RUN'S fingerprints, by run id. Files/parity also holds every earlier run's,
        # and a leg that failed this run must not be graded on a stale one.
        for engine in selected:
            if not notebookutils.fs.exists(fingerprint(engine)):
                raise RuntimeError(f"no fingerprint for {engine} in run {run_id}")
            Path(tmp, f"{engine}.json").write_text(notebookutils.fs.head(fingerprint(engine)))
        sh(sys.executable, ".github/scripts/parity.py", "compare", tmp)